# Redes Neuronales parte 1

## 1. Importar librerías generales

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import logging

## 2. Importar librerías TensorFlow

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds

## 3. Configurar el registro de TensorFlow

In [3]:
logger = tf.get_logger()
logger.setLevel (logging.ERROR)

## 4. Se obtienen los datos y metadatos del set

In [4]:
dataset, metadata = tfds.load('mnist', as_supervised=True, with_info=True)
train_dataset, test_dataset = dataset['train'], dataset['test']

## 5. Se agregan etiquetas de texto

In [5]:
class_names = [
'Cero', 'Uno', 'Dos', 'Tres', 'Cuatro', 'Cinco', 'Seis', 'Siete',
'Ocho', 'Nueve'
]

## 6. Se obtienen la cantidad de ejemplos en variables, para poder usarlos más adelante

In [6]:
num_train_examples = metadata.splits['train'].num_examples #60 mil datos train
num_test_examples = metadata.splits['test'].num_examples #10 mil datos test

## 7. Se normalizan los números de los pixeles de 0 a 255, para que sean de 0 a 1, es decir, 0 para blanco y 1 para negro. Sólo se divide el número entre 255.

In [7]:
#Normalizar: Numeros de 0 a 255, que sean de 0 a 1
def normalize(images, labels):
    images = tf.cast(images, tf.float32)
    images /= 255
    return images, labels

## 8. Normalizar los datos

In [8]:
train_dataset = train_dataset.map(normalize)
test_dataset = test_dataset.map(normalize)

## 9. Modelado

In [9]:
#Estructura de la red
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28,1)), #Capa de entrada
    tf.keras.layers.Dense(64, activation=tf.nn.relu), #Capas oculta
    tf.keras.layers.Dense(64, activation=tf.nn.relu), #Capas oculta
    tf.keras.layers.Dense(10, activation=tf.nn.softmax) #para clasificacion
])

/opt/anaconda3/envs/tf-env/lib/python3.11/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## 10. Indicamos que funciones se van a utilizar

In [10]:
#Función que compila el modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
#Aprendizaje por lotes de 32 cada lote
BATCHSIZE = 32
train_dataset = train_dataset.repeat().shuffle(num_train_examples).batch(BATCHSIZE)
test_dataset = test_dataset.batch(BATCHSIZE)

In [12]:
#Realizar el aprendizaje
model.fit(
    train_dataset, epochs=5,
    steps_per_epoch=math.ceil(num_train_examples/BATCHSIZE)
)

Epoch 1/5


2026-03-10 15:55:20.384898: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:376] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 806us/step - accuracy: 0.8625 - loss: 0.4852
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 805us/step - accuracy: 0.9579 - loss: 0.1399
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 778us/step - accuracy: 0.9706 - loss: 0.0978
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 814us/step - accuracy: 0.9753 - loss: 0.0765
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 802us/step - accuracy: 0.9817 - loss: 0.0622


# PARTE 2

In [13]:
from urllib import parse
from http.server import HTTPServer, BaseHTTPRequestHandler

In [14]:
class SimpleHTTPRequestHandler(BaseHTTPRequestHandler):

    def do_POST(self):
        print("Peticion recibida")
        
        #Obtener datos de la petición y limpiar los datos
        content_length = int(self.headers['Content-Length'])
        data = self.rfile.read(content_length)
        data = data.decode().replace('pixeles=', '')
        data = parse.unquote(data)
        
        #Realizar transformación para dejar igual que los ejemplos que usa MNIST
        arr = np.fromstring(data, np.float32, sep=",")
        arr = arr.reshape(28,28)
        arr = np.array(arr)
        arr = arr.reshape(1,28,28,1)
        
        #Realizar y obtener la predicción
        prediction_values = model.predict(arr, batch_size=1)
        prediction = str(np.argmax(prediction_values))
        print("Prediccion final: " + prediction)

        #Regresar respuesta a la peticion HTTP
        self.send_response(200)
        
        #Evitar problemas con CORS
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(prediction.encode())

## Iniciar el servidor

In [ ]:
#Iniciar el servidor en el puerto 8000 y escuchar por siempre
#Si se queda colgado, en el admon de tareas buscar la tarea de python y finalizar tarea
print("Iniciando el servidor...")
server = HTTPServer(('localhost', 8000), SimpleHTTPRequestHandler)
server.serve_forever()

Iniciando el servidor...
Peticion recibida
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Prediccion final: 5


127.0.0.1 - - [10/Mar/2026 15:55:39] "POST / HTTP/1.1" 200 -


Peticion recibida
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Prediccion final: 3


127.0.0.1 - - [10/Mar/2026 15:55:42] "POST / HTTP/1.1" 200 -


Peticion recibida
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Prediccion final: 1


127.0.0.1 - - [10/Mar/2026 15:55:47] "POST / HTTP/1.1" 200 -


Peticion recibida
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Prediccion final: 3


127.0.0.1 - - [10/Mar/2026 15:55:50] "POST / HTTP/1.1" 200 -
